# 7 · Distributed SeQUeNCe BB84 over FABRIC

Run real **SeQUeNCe** `QKDNode` / BB84 instances on two FABRIC nodes that pass
**real traffic on the wire**: photons cross as `0x7101` Ethernet frames through the
BMv2 **P4 switch** (which applies fiber loss), while classical sifting and QBER
**sample disclosure** ride raw `0x7102` Ethernet frames through the **same P4 switch** (no TCP) — the real-WAN lever.

This is the `qne-sequence/` runtime (see `qne-sequence/DESIGN.md`), the distributed
evolution of the hand-coded BB84 in notebook 2. The slice and data-plane IPs come from
notebook 1; only the **run** step differs (`qne_sequence.node_runner` instead of `qne.cli`).

**Both planes are raw L2 on FABRIC — no TCP:**
- **Quantum** — real `0x7101` photon frames through the BMv2 **P4 switch** (fiber loss in the data plane).
- **Classical** — sifting / QBER disclosure as raw `0x7102` frames through the **same switch** (`CLASSICAL='l2'`).
- Both share the photon interface, so `CLASSICAL='l2'` **requires** `TRANSPORT='raw'` + the switch (`LOSS` in `switch`/`auto`); run notebook 1 first.
- A TCP dev fallback exists in the backend but is **not** used here — this is the FABRIC configuration.

### At a glance
- **Purpose:** run one distributed-SeQUeNCe BB84 experiment across the slice and record the result.
- **Prereqs:** **run notebook 1 first** — it provisions the slice, starts BMv2, and sets up the data-plane IPs/ARP/promisc. This notebook reuses that slice.
- **Inputs:** `SLICE_NAME`, `SCENARIO` (match notebook 1), and the emulator knobs below.
- **Outputs:** `results/fabric_seq_{alice,bob}.json`; on-node logs at `/tmp/seq_{alice,bob}.log`.
- **Idempotent:** re-runnable — uploads code, builds a `.venv-qne` (sequence 1.0.0) on both nodes, re-arms the switch loss, then runs.

## 1 · Configuration

In [7]:
SLICE_NAME = 'qfabric-bb84-2'                          # same slice as notebook 1
SCENARIO   = 'validation/scenarios/fabric_1km.yml'     # sizes the P4 loss model + run params

# BMv2 image: must match notebook 1 (container path). '' => switch built from source.
BMV2_IMAGE = 'ghcr.io/kthare10/qfabric-bmv2:latest'

# --- Channels: FABRIC raw-L2 on BOTH planes, no TCP (see cell-1 notes) ---
TRANSPORT = 'raw'      # quantum photons as real 0x7101 frames through the BMv2 P4 switch
LOSS      = 'switch'   # 'switch' (BMv2 P4) | 'auto'  — L2 classical needs the switch in-path
CLASSICAL = 'l2'       # classical sifting/QBER as raw 0x7102 frames (NO TCP); needs TRANSPORT='raw' + switch

# Distributed-SeQUeNCe emulator knobs
NUM_PULSES      = 20000      # photons Alice emits
KEY_LENGTH      = 256        # target sifted key length (bits)
SAMPLE_FRACTION = 0.2        # fraction of sifted bits disclosed for QBER estimation
PHOTON_MODE     = 'bulk'     # 'bulk' (fast) or 'per_event' (per-photon fidelity) — see DESIGN §4.3
PHOTON_DRAIN_MS = 500        # wait for straggler photons after QUBITS_DONE (photon plane vs 0x7102 classical race)
PHOTON_RATE_HZ  = 10000      # raw bulk TX pacing (frames/s). 0 = unpaced burst — overruns
                             # BMv2/socket buffers and the drops masquerade as fiber loss.

## 2 · Load the slice

In [13]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

# deploy_fabric.py holds the tested provisioning/run logic; this notebook is a thin
# wrapper around it (single source of truth) — incl. the new SeQUeNCe-emulator helpers.
import deploy_fabric as df
from qne.config import ScenarioConfig

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show();
slice_obj.list_nodes();
slice_obj.list_interfaces();
slice_obj.list_networks();

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/Users/kthare10/work/fabric_config/id_token.json
Project ID,4604cab7-41ff-4c1a-a935-0ca6f20cceeb
Bastion Host,bastion.fabric-testbed.net
Bastion Username,kthare10_0011904101
Bastion Private Key File,/Users/kthare10/.ssh/bastion-prod-2
Slice Public Key File,/Users/kthare10/.ssh/id_rsa.pub
Slice Private Key File,/Users/kthare10/.ssh/id_rsa


User: kthare10@email.unc.edu bastion key is valid!
Configuration is valid


ID,9e082748-bdc9-4c91-b8b7-4b3753bf9f9c
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-10-04 02:31:48 +0000
Lease Start (UTC),2026-07-22 02:31:48 +0000
Project ID,4604cab7-41ff-4c1a-a935-0ca6f20cceeb
State,StableOK
Email,kthare10@email.unc.edu
UserId,43b7271b-90eb-45f6-833a-e51cf13bbc68


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
0555cecc-e13d-49f2-aec8-c7a0e3686512,alice,4,8,100,default_ubuntu_22,qcow2,tacc-w1.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:fe32:93e9,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fe32:93e9,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
b8456a2b-44a0-45b4-a911-3761a865ee93,bob,4,8,100,default_ubuntu_22,qcow2,tacc-w2.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:fe5c:35bf,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fe5c:35bf,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
3d360f9d-76d2-4dab-a7b7-cf8ba80ba049,switch,4,8,100,default_ubuntu_22,qcow2,tacc-w3.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:fee0:9b33,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fee0:9b33,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
alice-alice_nic-p1,p1,alice,net_alice_switch,100,,22:F5:5F:CE:C3:60,enp7s0,enp7s0,config,10.10.1.1,6,HundredGigE0/0/0/5
switch-sw_nic_alice-p1,p1,switch,net_alice_switch,100,,3A:84:1C:A5:5F:9C,enp8s0,enp8s0,config,fe80::3884:1cff:fea5:5f9c,4,HundredGigE0/0/0/9
switch-sw_nic_bob-p1,p1,switch,net_switch_bob,100,,3A:22:D7:38:CD:CD,enp7s0,enp7s0,config,fe80::3822:d7ff:fe38:cdcd,4,HundredGigE0/0/0/9
bob-bob_nic-p1,p1,bob,net_switch_bob,100,,12:E7:E6:F5:75:0C,enp7s0,enp7s0,config,10.10.1.2,6,HundredGigE0/0/0/7


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
385cfd55-89f0-41c7-b625-2ac4310c7d2d,net_alice_switch,L2,L2Bridge,TACC,None,None,Active,
3e15953e-c6d4-43d4-8deb-a738fc5f1164,net_switch_bob,L2,L2Bridge,TACC,None,None,Active,


## 3 · Ship the latest code + build the SeQUeNCe runtime

`upload_project` re-tars the repo to every node (this now includes `qne-sequence/`).
`setup_sequence_runtime` builds a dedicated `.venv-qne` (Python 3.12 + `sequence==1.0.0`
+ numpy) on **both** Alice and Bob and verifies the full import chain
(`sequence` + `qne` + `qne_sequence`). One-time per slice; idempotent (a few minutes).

In [15]:
df.upload_project(slice_obj)
if BMV2_IMAGE:
    os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE   # configure_switch uses the container
df.setup_sequence_runtime(slice_obj)


=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)

=== Setting up SeQUeNCe-emulator runtime (.venv-qne) on alice+bob ===
  [alice] python3.12 venv + sequence==1.0.0 ...
Reading package lists...
Building dependency tree...
Reading state information...
software-properties-common is already the newest version (0.99.22.9).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://nova.clouds.archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://nova.clouds.archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://nova.clouds.archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Reading package lists...
Repository: 'deb https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/ jammy main'
Description:

## 4 · Arm the P4 loss model (only when using the switch)

When `TRANSPORT='raw'` and `LOSS` is `switch`/`auto`, the fiber loss is the **switch's**
job: `configure_switch` compiles/starts BMv2 and installs the per-wavelength loss
threshold from the scenario. For `tcp`, `loss=model`, or `loss=none` **no switch is
needed** and this step is skipped. (Data-plane IPs/ARP/promisc come from notebook 1.)

In [16]:
cfg = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO)
USE_SWITCH = (TRANSPORT == 'raw' and LOSS in ('switch', 'auto'))
if USE_SWITCH:
    threshold = cfg.loss_threshold_u32
    print(f'Scenario {cfg.name}: distance={cfg.channel.distance_km} km, '
          f'atten={cfg.channel.attenuation_db_per_km} dB/km, '
          f'P(loss)={cfg.loss_probability:.4f}, P4 threshold(u32)={threshold}')
    df.configure_switch(slice_obj, threshold)   # (re)compile + start BMv2 + set loss table
else:
    print(f'No P4 switch needed (transport={TRANSPORT}, loss={LOSS}); skipping configure_switch.')

Scenario fabric_1km: distance=1.0 km, atten=0.2 dB/km, P(loss)=0.0450, P4 threshold(u32)=193305371

=== Configuring switch (threshold=193305371) ===
  Switch interfaces: enp8s0 (Alice), enp7s0 (Bob)
  Alice MAC: 22:F5:5F:CE:C3:60
  Bob MAC:   12:E7:E6:F5:75:0C
  Switch Alice-side MAC: 3A:84:1C:A5:5F:9C
  Switch Bob-side MAC:   3A:22:D7:38:CD:CD
  Using BMv2 Docker image: ghcr.io/kthare10/qfabric-bmv2:latest
  Compiling P4 (in container)...
  Starting BMv2 (container: --privileged --network host)...
  Configuring tables...
  Switch configured and running


## 5 · Run distributed-SeQUeNCe BB84

Bob listens (raw `0x7102` classical + `0x7101` photon RX); Alice connects over the data-plane IP and emits
photons per the chosen `TRANSPORT`/`LOSS`. Physics (fidelity/efficiency/dark counts) come
from the scenario; with `LOSS='none'` the channel is lossless (QBER from fidelity only).

In [17]:
a_res, b_res = df.run_sequence_bb84(
    slice_obj,
    transport=TRANSPORT,
    loss=LOSS,
    classical_transport=CLASSICAL,
    num_pulses=NUM_PULSES,
    key_length=KEY_LENGTH,
    fidelity=cfg.channel.polarization_fidelity,
    efficiency=cfg.detector.efficiency,
    dark_count_rate=cfg.detector.dark_count_rate,
    distance_km=cfg.channel.distance_km,
    attenuation=cfg.channel.attenuation_db_per_km,
    sample_fraction=SAMPLE_FRACTION,
    photon_mode=PHOTON_MODE,
    photon_drain_ms=PHOTON_DRAIN_MS,
    photon_rate_hz=PHOTON_RATE_HZ,
)


=== Running distributed-SeQUeNCe BB84 (raw 0x7101, loss=switch) ===
  Alice iface enp7s0 src 22:F5:5F:CE:C3:60 -> dst 3A:84:1C:A5:5F:9C
  Bob   iface enp7s0 src 12:E7:E6:F5:75:0C -> dst 3A:22:D7:38:CD:CD (L2 classical)
  Bob classical+photon TCP 10.10.1.2:5100 | pulses=20000 key_length=256 mode=bulk F=0.98 eff=0.8
  Starting Bob...
  Starting Alice (sender)...
  Waiting for Alice...

=== Distributed-SeQUeNCe BB84 Results ===
  [alice] transport=raw/l2 qber=0.01241830065359477 sifted=7654 reconciled=True corrections=51 leaked=613 secure_key_bits=4689 secure_fraction=0.8071415992183835 eve_fraction=0.0 key=yes remote_access_errors=0
  [bob] transport=raw/l2 qber=0.01241830065359477 sifted=7654 reconciled=True corrections=51 leaked=613 secure_key_bits=4689 secure_fraction=0.8071415992183835 eve_fraction=0.0 key=yes remote_access_errors=0
  keys match bit-for-bit: True


## 6 · Verify

In [18]:
analytical = (1.0 - cfg.channel.polarization_fidelity) / 2.0
print(f'analytical intrinsic QBER (1-F)/2 = {analytical:.4f}\n')

checks = []
def rec(name, ok, detail=''):
    checks.append(ok)
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f' — {detail}' if detail else ''))

rec('alice produced a key', bool(a_res) and a_res.get('key') is not None)
rec('bob produced a key',   bool(b_res) and b_res.get('key') is not None)
if a_res:
    rec('alice: zero remote-access errors', a_res.get('remote_access_errors') == 0,
        f"errs={a_res.get('remote_access_errors')}")
    q = a_res.get('qber')
    rec('QBER plausible (< 0.11)', q is not None and 0 <= q < 0.11, f'QBER={q}')
    rec('sifted bits > key length', (a_res.get('sifted_bits') or 0) > KEY_LENGTH,
        f"sifted={a_res.get('sifted_bits')}")
    rec('secure fraction > 0', (a_res.get('secure_fraction') or 0) > 0,
        f"secure_fraction={a_res.get('secure_fraction')}")

# --- cross-node consistency (would catch head-race photon loss / desync) ---
if a_res and b_res:
    sa, sb = a_res.get('sifted_bits'), b_res.get('sifted_bits')
    rec('alice/bob agree on sifted count',
        sa is not None and sa == sb, f"alice={sa} bob={sb}")
    ka, kb = a_res.get('key_bits'), b_res.get('key_bits')
    rec('alice/bob agree on key length (bits)',
        ka is not None and ka == kb,   # None==None must NOT pass (missing field)
        f"alice={ka} bob={kb}")

    # Without error correction (no Cascade yet) the keys differ ONLY on the
    # error positions — the mismatch fraction must track the measured QBER.
    # A desynced run (lost head-of-train, wrong indices) shows ~50% mismatch.
    if a_res.get('key') is not None and b_res.get('key') is not None:
        import math
        q = a_res.get('qber') or 0.0
        nbits = KEY_LENGTH
        err_frac = bin(int(a_res['key']) ^ int(b_res['key'])).count('1') / nbits
        if q == 0:
            rec('keys identical (QBER = 0)', err_frac == 0.0, f'mismatch={err_frac:.4f}')
        else:
            bound = q + 4 * math.sqrt(q * (1 - q) / nbits) + 1 / nbits
            rec('key mismatch tracks QBER (no Cascade yet)', err_frac <= bound,
                f'mismatch={err_frac:.4f} vs QBER={q:.4f} (bound {bound:.4f})')

    # Photon accounting: measured sift vs the model. A big shortfall means
    # photons are being lost OUTSIDE the model — unpaced TX bursts overrunning
    # BMv2/socket buffers, head-of-train race, or switch misconfig.
    loss_p = a_res.get('loss_probability')
    if loss_p is not None and LOSS != 'none':
        expected = NUM_PULSES * (1 - loss_p) * cfg.detector.efficiency * 0.5
        measured = a_res.get('sifted_bits') or 0
        ratio = measured / expected if expected else 0.0
        rec('sifted count consistent with loss model (ratio in [0.7, 1.3])',
            0.7 <= ratio <= 1.3,
            f'measured={measured} expected~{expected:.0f} ratio={ratio:.2f}')

print(f"\n(transport={a_res.get('quantum_transport') if a_res else '?'}, "
      f"loss_where={a_res.get('loss_where') if a_res else '?'})")
print('\nALL CHECKS PASSED' if checks and all(checks)
      else '\nSOME CHECKS FAILED — inspect /tmp/seq_{alice,bob}.log on the nodes'
           + (' and `sudo docker logs bmv2` (or journalctl -u bmv2) on the switch.' if USE_SWITCH else '.'))

analytical intrinsic QBER (1-F)/2 = 0.0100

  [PASS] alice produced a key
  [PASS] bob produced a key
  [PASS] alice: zero remote-access errors — errs=0
  [PASS] QBER plausible (< 0.11) — QBER=0.01241830065359477
  [PASS] sifted bits > key length — sifted=7654
  [PASS] secure fraction > 0 — secure_fraction=0.8071415992183835
  [PASS] alice/bob agree on sifted count — alice=7654 bob=7654
  [PASS] alice/bob agree on key length (bits) — alice=6124 bob=6124
  [PASS] key mismatch tracks QBER (no Cascade yet) — mismatch=0.0000 vs QBER=0.0124 (bound 0.0440)
  [PASS] sifted count consistent with loss model (ratio in [0.7, 1.3]) — measured=7654 expected~7640 ratio=1.00

(transport=raw, loss_where=switch)

ALL CHECKS PASSED


## 7 · Notes & cleanup

- The slice stays up for re-runs. Sweep distance/fidelity by editing `SCENARIO` (or
  the knobs in cell 1) and re-running cells 4–6.
- **Tuning:** if the *photon accounting* check fails low (measured ≪ expected), lower
  `PHOTON_RATE_HZ` (BMv2/socket buffers are dropping frames) or raise `PHOTON_DRAIN_MS`
  (photons still in flight when `QUBITS_DONE` arrived over the `0x7102` classical channel). Photons that arrive
  *before* `BEGIN_PHOTON_PULSE` are buffered and replayed automatically (head-race fix).
  Try `PHOTON_MODE='per_event'` for per-photon timing fidelity (slower).
- **Key mismatch:** without Cascade (Phase D) Alice's and Bob's keys differ on the error
  positions by design — the verify cell checks the mismatch fraction *tracks the QBER*,
  which distinguishes expected errors from a desynced run (~50% mismatch).
- **Classical-network effects:** `df.apply_classical_netem(slice_obj, classical_transport='l2', delay_ms=..., loss_pct=...)`
  impairs only the raw `0x7102` classical channel (photon plane untouched) — the QFabric research lever, here over
  the SeQUeNCe stack. Clear with `df.clear_classical_netem(slice_obj)`.
- **Delete the slice** when done: `df.cleanup(fablib, SLICE_NAME)`.